# 06 — Figures: Simonyi night time by category, Oct 2025 → Jul 2026

One per-night table (`data/processed/per_night.csv`) feeds every figure. Categories per night, summing to the −18° night:
**on sky**, **weather** (WEATHER bit from 2026-02-25; EFD criteria + night-report cloud before), **non-weather loss**
(fault + idle, reported as "fault" for the talks). Colours from the repo palette via `time_loss.style`.

Outputs in `../figures/`: `time_loss_nightly_<tag>.png` (per-night stacked bars), `time_loss_monthly_<tag>.png` (monthly
fractions), each in a `_talk` (16:9, large type) and a `_paper` (AAS column) version. `<tag>` = day_obs range.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, DAY_OBS_WEATHER_BIT_START, GAP_MIN_MINUTES
from time_loss.style import COLORS, rp
PROC = Path("../data/processed"); FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
TAG = f"{DAY_OBS_MIN}-{DAY_OBS_MAX}"
t = pd.read_csv(PROC / "per_night.csv"); t["date"] = pd.to_datetime(t.day_obs.astype(str))
t["month"] = t.day_obs // 100
t["survey_plot"] = t.on_sky_hours + t.startup_hours  # A1+A2 (+A3 start-up allowance)
t["fault_plot"] = t.nonweather_loss_hours + t.unattributed_hours.fillna(0)  # D+E (+F from 2026-02-25), one series on the plots
tot = t[["night_hours", "survey_plot", "engineering_hours", "weather_hours", "fault_plot"]].sum()
print(tot.round(1).to_dict()); print({k: f"{v / tot.night_hours:.1%}" for k, v in tot.items() if k != "night_hours"})
# ROSP-24 categories, stack bottom → top = reversed precedence (science at the bottom)
CAT = [("survey_plot", "survey (on sky)", COLORS["on_sky"]), ("fault_plot", "fault + idle", COLORS["nonweather_loss"]), ("engineering_hours", "engineering", COLORS["engineering"]), ("weather_hours", "weather", COLORS["weather"])]

In [ ]:
def rc(kind):
    if kind == "talk":
        return {"figure.figsize": (13.33, 7.5), "font.size": 18, "axes.labelsize": 20, "axes.titlesize": 22, "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 17, "axes.linewidth": 1.2}
    return {"figure.figsize": (7.0, 3.6), "font.size": 8, "axes.labelsize": 8.5, "axes.titlesize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "axes.linewidth": 0.6, "pdf.fonttype": 42}

def nightly(kind, transparent=False):
    with mpl.rc_context(rc(kind)):
        fig, ax = plt.subplots()
        bottom = np.zeros(len(t))
        for col, lab, c in CAT:
            ax.bar(t.date, t[col], bottom=bottom, width=1.0, color=c, label=lab, linewidth=0)
            bottom += t[col].to_numpy()
        ax.plot(t.date, t.night_hours, color=COLORS["ink"], lw=0.8, label="night (−12°)")
        xb = pd.Timestamp(str(DAY_OBS_WEATHER_BIT_START))
        ax.axvline(xb, color=COLORS["ink_secondary"], lw=1, ls="--"); ax.text(xb, ax.get_ylim()[1] * 0.98, " WEATHER bit →", va="top", ha="left", color=COLORS["ink_secondary"], fontsize=rc(kind)["legend.fontsize"] * 0.8)
        ax.set_ylabel("hours per night"); ax.set_xlim(t.date.min() - pd.Timedelta(days=1), t.date.max() + pd.Timedelta(days=1))
        ax.legend(frameon=False, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12))
        ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
        share = {lab: t[col].sum() / tot.night_hours for col, lab, _ in CAT}
        ax.set_title(f"Simonyi night time, {t.date.min():%d %b %Y} – {t.date.max():%d %b %Y}\n" + ", ".join(f"{v:.0%} {k}" for k, v in share.items()) + f" (of {tot.night_hours:.0f} h between −12° twilights)", loc="left")
        fig.tight_layout()
        fig.savefig(FIG / f"time_loss_nightly_{kind}_{TAG}.png", dpi=200, transparent=transparent)
        return fig
nightly("talk"); nightly("paper");

In [ ]:
mo = t.groupby("month")[["night_hours", "survey_plot", "engineering_hours", "weather_hours", "fault_plot"]].sum()
mo_n = t.groupby("month").size()
def monthly(kind, transparent=False):
    with mpl.rc_context(rc(kind)):
        fig, ax = plt.subplots()
        x = np.arange(len(mo)); bottom = np.zeros(len(mo))
        for col, lab, c in CAT:
            frac = (mo[col] / mo.night_hours).to_numpy()
            ax.bar(x, frac, bottom=bottom, color=c, label=lab, width=0.8, linewidth=0)
            for xi, f, b in zip(x, frac, bottom):
                if f > 0.06: ax.text(xi, b + f / 2, f"{f:.0%}", ha="center", va="center", color="white", fontsize=rc(kind)["xtick.labelsize"])
            bottom += frac
        ax.set_xticks(x); ax.set_xticklabels([pd.Timestamp(f"{m // 100}-{m % 100:02d}-01").strftime("%b\n%Y") + f"\n({n} n)" for m, n in zip(mo.index, mo_n)])
        ax.set_ylim(0, 1); ax.set_ylabel("fraction of −12° night"); ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
        ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2)); ax.spines[["top", "right"]].set_visible(False)
        ax.set_title("Simonyi night time by month (partial months: Oct 2025, Jul 2026)")
        fig.tight_layout(); fig.savefig(FIG / f"time_loss_monthly_{kind}_{TAG}.png", dpi=200, transparent=transparent)
        return fig
monthly("talk"); monthly("paper");
(mo.assign(n=mo_n).round(1))

## gap_min sensitivity (for the caption)

In [ ]:
sens = pd.read_csv("../data/step0/visit_coverage_gapmin_2_5_10.csv")
s = sens[["on_sky_h_gap2", "on_sky_h_gap5", "on_sky_h_gap10"]].sum().round(1)
print("on-sky hours total at gap_min 2 / 5 / 10 min:", s.tolist(), "| non-weather loss moves by the same amounts in the opposite direction; weather hours are unaffected")